# ReturnIQ: Exploratory Data Analysis (EDA)

## 3. Exploratory Analysis

This notebook analyses the cleaned and engineered ReturnIQ dataset.

The analysis focuses on:

- Dataset structure
- Order status
- Delivery performance and delay
- Customer review scores
- Product categories
- Seller performance
- Price and freight
- Seller-customer distance
- Transit speed
- Single-item vs multi-item orders
- Correlations and feature relationships
- External complaint evidence from NCH
- Return/cancellation signals from Online Retail II

EDA findings describe observed patterns and associations. They should not be interpreted as causal relationships.

## 3.1 Load the Cleaned Analytical Dataset

In [1]:
%run 01_Data_Loading.ipynb
%run 02_Data_Cleaning.ipynb
import pandas as pd
import numpy as np

# Load the integrated dataset created by 02_Data_Cleaning.ipynb.
df = pd.read_csv("data/cleaned/returniq_integrated_features.csv")

# Restore datetime columns used in EDA calculations.
datetime_columns = [
    "shipping_limit_date",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in datetime_columns:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors="coerce")

order_items_complete = df

# Load cleaned external datasets used in the final EDA.
df_complaints_clean = pd.read_csv("data/cleaned/complaints_cleaned.csv")
df_online_retail_clean = pd.read_csv("data/cleaned/online_retail_cleaned.csv")

print("Integrated dataset:", order_items_complete.shape)
print("Complaint dataset:", df_complaints_clean.shape)
print("Online Retail II:", df_online_retail_clean.shape)


Customers: 99441 rows, 5 columns
Geolocation: 1000163 rows, 5 columns
Order Items: 112650 rows, 7 columns
Payments: 103886 rows, 5 columns
Reviews: 99224 rows, 7 columns
Orders: 99441 rows, 8 columns
Products: 32951 rows, 9 columns
Sellers: 3095 rows, 4 columns
Translation: 71 rows, 2 columns
Online Retail II: 525461 rows, 8 columns
Customers
Rows: 99441
Columns: 5
Column names:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Geolocation
Rows: 1000163
Columns: 5
Column names:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Order Items
Rows: 112650
Columns: 7
Column names:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Payments
Rows: 103886
Columns: 5
Column names:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Reviews
Rows: 99224
Columns: 7
Column names:
['review_id

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


Quantity statistics:
count    525461.000000
mean         10.337667
std         107.424110
min       -9600.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       19152.000000
Name: Quantity, dtype: float64

Price statistics:
count    525461.000000
mean          4.688834
std         146.126914
min      -53594.360000
25%           1.250000
50%           2.100000
75%           4.210000
max       25111.090000
Name: Price, dtype: float64
Negative Quantity rows: 12326
Negative Price rows: 3

Rows with negative Quantity:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom



Rows with negative Price:


...

Review aggregated shape: (98673, 5)


,order_id,review_score_mean,review_score_min,review_score_count,review_comments
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,5,1,"Perfeito, produto entregue antes do combinado."
1,00018f77f2f0320c557190d7a144bdd3,4.0,4,1,
2,000229ec398224ef6ca0657da4fc703e,5.0,5,1,Chegou antes do prazo previsto e o produto sur...
3,00024acbcdf0a6daa1e931b038114c75,4.0,4,1,
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,5,1,Gostei pois veio no prazo determinado .


Payment aggregated shape: (99440, 5)


,order_id,payment_total_value,payment_count,payment_max_installments,payment_types
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3,credit_card


Products for merge: (32951, 6)
Sellers for merge: (3095, 4)
Order Items shape: (112650, 7)
After Products merge: (112650, 12)
After Sellers merge: (112650, 15)
Orders for merge: (99441, 8)
Before Orders merge: (112650, 15)
After Orders merge: (112650, 22)
Customers for merge: (99441, 5)
Before Customers merge: (112650, 22)
After Customers merge: (112650, 26)
Before Payments merge: (112650, 26)
After Payments merge: (112650, 30)
Before Reviews merge: (112650, 30)
After Reviews merge: (112650, 34)
Shape: (112650, 34)

Columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id',

## 3.2 Dataset Overview

In [2]:
# ============================================
# STEP 32.1 — FINAL DATASET OVERVIEW
# ============================================

print("========== DATASET OVERVIEW ==========")

print("Rows:", order_items_complete.shape[0])
print("Columns:", order_items_complete.shape[1])

print("\nDataset memory usage:")
print(
    f"{order_items_complete.memory_usage(deep=True).sum() / (1024**2):.2f} MB"
)

print("\n========== DATA TYPES ==========")
print(order_items_complete.dtypes.value_counts())

print("\n========== MISSING VALUES ==========")

missing_summary = (
    order_items_complete.isnull()
    .sum()
    .reset_index()
)

missing_summary.columns = ["column", "missing_count"]

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"]
    / len(order_items_complete)
    * 100
)

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values(
    by="missing_count",
    ascending=False
)

print(missing_summary.to_string(index=False))

========== DATASET OVERVIEW ==========
Rows: 112650
Columns: 71

Dataset memory usage:
92.14 MB

========== DATA TYPES ==========
float64           28
int64             19
str               18
datetime64[us]     6
Name: count, dtype: int64

========== MISSING VALUES ==========
                       column  missing_count  missing_percentage
              review_comments          64237           57.023524
    shipping_preparation_days           2760            2.450067
                 transit_days           2691            2.388815
                delivery_days           2454            2.178429
          delivery_delay_days           2454            2.178429
order_delivered_customer_date           2454            2.178429
                   is_delayed           2454            2.178429
 order_delivered_carrier_date           1381            1.225921
            review_score_mean            942            0.836218
             review_score_min            942            0.836218
       

## 2. EDA Analysis

In [3]:
# ============================================
# STEP 32.2 — ORDER STATUS ANALYSIS
# ============================================

import matplotlib.pyplot as plt

# Count orders by status
order_status_counts = (
    order_items_complete["order_status"]
    .value_counts()
)

print("========== ORDER STATUS COUNTS ==========")
print(order_status_counts)

print("\n========== ORDER STATUS PERCENTAGE ==========")

order_status_percentage = (
    order_status_counts
    / order_status_counts.sum()
    * 100
)

print(order_status_percentage.round(2))

========== ORDER STATUS COUNTS ==========
order_status
delivered      110197
shipped          1185
canceled          542
invoiced          359
processing        357
unavailable         7
approved            3
Name: count, dtype: int64

========== ORDER STATUS PERCENTAGE ==========
order_status
delivered      97.82
shipped         1.05
canceled        0.48
invoiced        0.32
processing      0.32
unavailable     0.01
approved        0.00
Name: count, dtype: float64


## 3. EDA Analysis

In [4]:
# ============================================
# STEP 32.3 — DELIVERY PERFORMANCE ANALYSIS
# ============================================

print("========== DELIVERY DAYS SUMMARY ==========")

print(
    order_items_complete["delivery_days"]
    .describe()
)

print("\n========== ESTIMATED DELIVERY DAYS SUMMARY ==========")

print(
    order_items_complete["estimated_delivery_days"]
    .describe()
)

print("\n========== DELIVERY DELAY SUMMARY ==========")

print(
    order_items_complete["delivery_delay_days"]
    .describe()
)

print("\n========== DELAY STATUS ==========")

print(
    order_items_complete["is_delayed"]
    .value_counts(dropna=False)
)

========== DELIVERY DAYS SUMMARY ==========
count    110196.000000
mean         12.007723
std           9.451455
min           0.000000
25%           6.000000
50%          10.000000
75%          15.000000
max         209.000000
Name: delivery_days, dtype: float64

========== ESTIMATED DELIVERY DAYS SUMMARY ==========
count    112650.000000
mean         23.470484
std           8.884872
min           2.000000
25%          18.000000
50%          23.000000
75%          28.000000
max         155.000000
Name: estimated_delivery_days, dtype: float64

========== DELIVERY DELAY SUMMARY ==========
count    110196.000000
mean        -12.030201
std          10.160157
min        -147.000000
25%         -17.000000
50%         -13.000000
75%          -7.000000
max         188.000000
Name: delivery_delay_days, dtype: float64

========== DELAY STATUS ==========
is_delayed
0.0    102931
1.0      7265
NaN      2454
Name: count, dtype: int64


## 4. EDA Analysis

In [5]:
# ============================================
# STEP 32.4 — DELIVERY DELAY ANALYSIS
# ============================================

# Separate valid delivery delay values
delay_data = order_items_complete[
    order_items_complete["delivery_delay_days"].notna()
]["delivery_delay_days"]

print("========== EARLY / ON-TIME / DELAYED ==========")

print(
    "Early delivery:",
    (delay_data < 0).sum()
)

print(
    "On estimated date:",
    (delay_data == 0).sum()
)

print(
    "Delayed:",
    (delay_data > 0).sum()
)

print("\n========== DELAY PERCENTAGE ==========")

total_valid = len(delay_data)

print(
    f"Early delivery: {(delay_data < 0).sum() / total_valid * 100:.2f}%"
)

print(
    f"On estimated date: {(delay_data == 0).sum() / total_valid * 100:.2f}%"
)

print(
    f"Delayed: {(delay_data > 0).sum() / total_valid * 100:.2f}%"
)

print("\n========== ACTUAL DELAY ONLY ==========")

actual_delays = delay_data[delay_data > 0]

print(actual_delays.describe())

========== EARLY / ON-TIME / DELAYED ==========
Early delivery: 101481
On estimated date: 1450
Delayed: 7265

========== DELAY PERCENTAGE ==========
Early delivery: 92.09%
On estimated date: 1.32%
Delayed: 6.59%

========== ACTUAL DELAY ONLY ==========
count    7265.000000
mean       10.488782
std        14.499579
min         1.000000
25%         3.000000
50%         7.000000
75%        13.000000
max       188.000000
Name: delivery_delay_days, dtype: float64


## 5. EDA Analysis

In [6]:
# ============================================
# STEP 32.5 — REVIEW SCORE ANALYSIS
# ============================================

print("========== REVIEW SCORE COUNTS ==========")

review_score_counts = (
    order_items_complete["review_score_mean"]
    .value_counts()
    .sort_index()
)

print(review_score_counts)

print("\n========== REVIEW SCORE PERCENTAGE ==========")

review_score_percentage = (
    review_score_counts
    / review_score_counts.sum()
    * 100
)

print(review_score_percentage.round(2))

print("\n========== REVIEW SCORE SUMMARY ==========")

print(
    order_items_complete["review_score_mean"]
    .describe()
)

========== REVIEW SCORE COUNTS ==========
review_score_mean
1.000000    14079
1.500000       18
2.000000     3833
2.500000       45
3.000000     9384
3.333333        1
3.500000       31
4.000000    21166
4.333333        1
4.500000       68
5.000000    63082
Name: count, dtype: int64

========== REVIEW SCORE PERCENTAGE ==========
review_score_mean
1.000000    12.60
1.500000     0.02
2.000000     3.43
2.500000     0.04
3.000000     8.40
3.333333     0.00
3.500000     0.03
4.000000    18.95
4.333333     0.00
4.500000     0.06
5.000000    56.47
Name: count, dtype: float64

========== REVIEW SCORE SUMMARY ==========
count    111708.000000
mean          4.033128
std           1.386119
min           1.000000
25%           4.000000
50%           5.000000
75%           5.000000
max           5.000000
Name: review_score_mean, dtype: float64


## 6. EDA Analysis

In [7]:
# ============================================
# STEP 32.6 — REVIEW SCORE VS DELIVERY DELAY
# ============================================

# Create a simple review group
order_items_complete["review_group"] = pd.cut(
    order_items_complete["review_score_mean"],
    bins=[0, 2, 3, 4, 5],
    labels=[
        "Low (1-2)",
        "Average (3)",
        "Good (4)",
        "Excellent (5)"
    ],
    include_lowest=True
)

print("========== REVIEW GROUP COUNTS ==========")

print(
    order_items_complete["review_group"]
    .value_counts()
    .sort_index()
)

print("\n========== AVERAGE DELIVERY DELAY BY REVIEW GROUP ==========")

review_delay_analysis = (
    order_items_complete
    .groupby("review_group", observed=True)
    .agg(
        order_items=("order_id", "count"),
        average_delay_days=("delivery_delay_days", "mean"),
        median_delay_days=("delivery_delay_days", "median"),
        delayed_items=("is_delayed", "sum")
    )
)

review_delay_analysis["delayed_percentage"] = (
    review_delay_analysis["delayed_items"]
    / review_delay_analysis["order_items"]
    * 100
)

print(
    review_delay_analysis.round(2)
)

========== REVIEW GROUP COUNTS ==========
review_group
Low (1-2)        17930
Average (3)       9429
Good (4)         21198
Excellent (5)    63151
Name: count, dtype: int64

========== AVERAGE DELIVERY DELAY BY REVIEW GROUP ==========
               order_items  average_delay_days  median_delay_days  \
review_group                                                        
Low (1-2)            17930               -6.79               -9.0   
Average (3)           9429              -11.08              -11.0   
Good (4)             21198              -12.48              -12.0   
Excellent (5)        63151              -13.42              -13.0   

               delayed_items  delayed_percentage  
review_group                                      
Low (1-2)             4454.0               24.84  
Average (3)            752.0                7.98  
Good (4)               708.0                3.34  
Excellent (5)         1170.0                1.85  


## 7. EDA Analysis

In [8]:
# ============================================
# STEP 32.7 — REMOVE TEMPORARY EDA COLUMN
# ============================================

if "review_group" in order_items_complete.columns:
    order_items_complete.drop(
        columns=["review_group"],
        inplace=True
    )

print("review_group removed successfully.")

print("\nCurrent dataset shape:")
print(order_items_complete.shape)

review_group removed successfully.

Current dataset shape:
(112650, 71)


## 8. EDA Analysis

In [9]:
# ============================================
# STEP 32.8 — PRODUCT CATEGORY ANALYSIS
# ============================================

print("========== PRODUCT CATEGORY OVERVIEW ==========")

# Number of unique product categories
unique_categories = (
    order_items_complete["product_category_name_english"]
    .nunique()
)

print("Number of unique product categories:", unique_categories)

# Count order items by category
category_counts = (
    order_items_complete["product_category_name_english"]
    .value_counts()
)

print("\n========== TOP 20 PRODUCT CATEGORIES ==========")
print(category_counts.head(20))

# Percentage of order items
category_percentage = (
    category_counts
    / len(order_items_complete)
    * 100
)

print("\n========== TOP 20 CATEGORY PERCENTAGES ==========")
print(category_percentage.head(20).round(2))

========== PRODUCT CATEGORY OVERVIEW ==========
Number of unique product categories: 74

========== TOP 20 PRODUCT CATEGORIES ==========
product_category_name_english
bed_bath_table              11115
health_beauty                9670
sports_leisure               8641
furniture_decor              8334
computers_accessories        7827
housewares                   6964
watches_gifts                5991
telephony                    4545
garden_tools                 4347
auto                         4235
toys                         4117
cool_stuff                   3796
perfumery                    3419
baby                         3065
electronics                  2767
stationery                   2517
fashion_bags_accessories     2031
pet_shop                     1947
office_furniture             1691
Unknown                      1603
Name: count, dtype: int64

========== TOP 20 CATEGORY PERCENTAGES ==========
product_category_name_english
bed_bath_table              9.87
health_beauty

## 9. EDA Analysis

In [10]:
# ============================================
# STEP 32.9 — CATEGORY VS CUSTOMER EXPERIENCE
# ============================================

category_experience = (
    order_items_complete
    .groupby("product_category_name_english", observed=True)
    .agg(
        order_items=("order_id", "count"),
        average_review_score=("review_score_mean", "mean"),
        delayed_items=("is_delayed", "sum"),
        average_delivery_days=("delivery_days", "mean")
    )
    .reset_index()
)

# Calculate delay percentage
category_experience["delayed_percentage"] = (
    category_experience["delayed_items"]
    / category_experience["order_items"]
    * 100
)

# Sort by number of orders
category_experience = category_experience.sort_values(
    by="order_items",
    ascending=False
)

print("========== CATEGORY VS CUSTOMER EXPERIENCE ==========")
print(
    category_experience
    .head(20)
    .round(2)
    .to_string(index=False)
)

print("\n========== CATEGORIES WITH HIGHEST DELAY PERCENTAGE ==========")

# Only consider categories with at least 100 order items
reliable_delay_categories = category_experience[
    category_experience["order_items"] >= 100
]

print(
    reliable_delay_categories
    .sort_values(
        by="delayed_percentage",
        ascending=False
    )
    .head(15)
    .round(2)
    .to_string(index=False)
)

========== CATEGORY VS CUSTOMER EXPERIENCE ==========
product_category_name_english  order_items  average_review_score  delayed_items  average_delivery_days  delayed_percentage
               bed_bath_table        11115                  3.90          770.0                  12.34                6.93
                health_beauty         9670                  4.14          717.0                  11.52                7.41
               sports_leisure         8641                  4.11          532.0                  11.69                6.16
              furniture_decor         8334                  3.91          574.0                  12.40                6.89
        computers_accessories         7827                  3.93          496.0                  12.78                6.34
                   housewares         6964                  4.05          340.0                  10.46                4.88
                watches_gifts         5991                  4.02          422.0      

## 10. EDA Analysis

In [11]:
# ============================================
# STEP 32.10 — SELLER PERFORMANCE ANALYSIS
# ============================================

seller_performance = (
    order_items_complete
    .groupby("seller_id")
    .agg(
        order_items=("order_id", "count"),
        average_review_score=("review_score_mean", "mean"),
        delayed_items=("is_delayed", "sum"),
        average_delivery_days=("delivery_days", "mean"),
        average_freight_value=("freight_value", "mean")
    )
    .reset_index()
)

# Calculate delay percentage
seller_performance["delayed_percentage"] = (
    seller_performance["delayed_items"]
    / seller_performance["order_items"]
    * 100
)

print("========== SELLER PERFORMANCE OVERVIEW ==========")
print("Number of sellers:", len(seller_performance))

print("\n========== TOP 15 SELLERS BY ORDER VOLUME ==========")
print(
    seller_performance
    .sort_values("order_items", ascending=False)
    .head(15)
    .round(2)
    .to_string(index=False)
)

# Avoid unreliable percentages from sellers with very few orders
reliable_sellers = seller_performance[
    seller_performance["order_items"] >= 50
]

print("\n========== TOP 15 SELLERS BY DELAY PERCENTAGE ==========")
print(
    reliable_sellers
    .sort_values("delayed_percentage", ascending=False)
    .head(15)
    .round(2)
    .to_string(index=False)
)

print("\n========== SELLERS WITH LOWEST REVIEW SCORES ==========")
print(
    reliable_sellers
    .sort_values("average_review_score", ascending=True)
    .head(15)
    .round(2)
    .to_string(index=False)
)

========== SELLER PERFORMANCE OVERVIEW ==========
Number of sellers: 3095

========== TOP 15 SELLERS BY ORDER VOLUME ==========
                       seller_id  order_items  average_review_score  delayed_items  average_delivery_days  average_freight_value  delayed_percentage
6560211a19b47992c3666cc44a7e94c0         2033                  3.91          102.0                   9.06                  13.75                5.02
4a3ca9315b744ce9f8e9374361493884         1987                  3.80          189.0                  13.94                  17.65                9.51
1f50f920176fa81dab994f9023523100         1931                  3.99          150.0                  15.11                  18.21                7.77
cc419e0650a3c5ba77189a1882b7556a         1775                  4.07           88.0                  11.06                  14.46                4.96
da8622b14eb17ae2831f4ac5b9dab84a         1551                  4.06           99.0                  10.70                  16.0

## 11. EDA Analysis

In [12]:
# ============================================
# STEP 32.11 — PRICE & FREIGHT ANALYSIS
# ============================================

print("========== PRICE SUMMARY ==========")
print(order_items_complete["price"].describe().round(2))

print("\n========== FREIGHT VALUE SUMMARY ==========")
print(order_items_complete["freight_value"].describe().round(2))

print("\n========== FREIGHT TO PRICE RATIO ==========")
print(
    order_items_complete["freight_to_price_ratio"]
    .describe()
    .round(2)
)

print("\n========== HIGH FREIGHT RATIO ==========")
print(
    order_items_complete["high_freight_ratio"]
    .value_counts()
)

print("\n========== HIGH FREIGHT RATIO PERCENTAGE ==========")
print(
    (
        order_items_complete["high_freight_ratio"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)

# Compare delivery delay by freight ratio group
freight_analysis = (
    order_items_complete
    .groupby("high_freight_ratio", observed=True)
    .agg(
        order_items=("order_id", "count"),
        average_price=("price", "mean"),
        average_freight=("freight_value", "mean"),
        average_freight_ratio=("freight_to_price_ratio", "mean"),
        average_delivery_days=("delivery_days", "mean"),
        delayed_items=("is_delayed", "sum")
    )
)

freight_analysis["delayed_percentage"] = (
    freight_analysis["delayed_items"]
    / freight_analysis["order_items"]
    * 100
)

print("\n========== FREIGHT RATIO VS DELIVERY ==========")
print(freight_analysis.round(2))

========== PRICE SUMMARY ==========
count    112650.00
mean        120.65
std         183.63
min           0.85
25%          39.90
50%          74.99
75%         134.90
max        6735.00
Name: price, dtype: float64

========== FREIGHT VALUE SUMMARY ==========
count    112650.00
mean         19.99
std          15.81
min           0.00
25%          13.08
50%          16.26
75%          21.15
max         409.68
Name: freight_value, dtype: float64

========== FREIGHT TO PRICE RATIO ==========
count    112650.00
mean          0.32
std           0.35
min           0.00
25%           0.13
50%           0.23
75%           0.39
max          26.24
Name: freight_to_price_ratio, dtype: float64

========== HIGH FREIGHT RATIO ==========
high_freight_ratio
0    93706
1    18944
Name: count, dtype: int64

========== HIGH FREIGHT RATIO PERCENTAGE ==========
high_freight_ratio
0    83.18
1    16.82
Name: proportion, dtype: float64

========== FREIGHT RATIO VS DELIVERY ==========
                    ord

## 12. EDA Analysis

In [13]:
# ============================================
# STEP 32.12 — DISTANCE & LOGISTICS ANALYSIS
# ============================================

print("========== DISTANCE SUMMARY ==========")

print(
    order_items_complete["seller_customer_distance_km"]
    .describe()
    .round(2)
)

print("\n========== DISTANCE CATEGORY COUNTS ==========")

distance_counts = (
    order_items_complete["distance_category"]
    .value_counts()
)

print(distance_counts)

print("\n========== DISTANCE CATEGORY PERCENTAGE ==========")

print(
    (
        distance_counts
        / len(order_items_complete)
        * 100
    ).round(2)
)

# Compare logistics performance by distance category
distance_analysis = (
    order_items_complete
    .groupby("distance_category", observed=True)
    .agg(
        order_items=("order_id", "count"),
        average_distance_km=("seller_customer_distance_km", "mean"),
        average_delivery_days=("delivery_days", "mean"),
        delayed_items=("is_delayed", "sum"),
        average_freight=("freight_value", "mean"),
        average_review_score=("review_score_mean", "mean")
    )
    .reset_index()
)

distance_analysis["delayed_percentage"] = (
    distance_analysis["delayed_items"]
    / distance_analysis["order_items"]
    * 100
)

print("\n========== DISTANCE VS LOGISTICS ==========")

print(
    distance_analysis
    .round(2)
    .to_string(index=False)
)

========== DISTANCE SUMMARY ==========
count    112096.00
mean        597.00
std         590.04
min           0.00
25%         184.11
50%         431.75
75%         792.25
max        8677.92
Name: seller_customer_distance_km, dtype: float64

========== DISTANCE CATEGORY COUNTS ==========
distance_category
Very Far    38564
Far         36958
Near        20993
Medium      15581
Unknown       554
Name: count, dtype: int64

========== DISTANCE CATEGORY PERCENTAGE ==========
distance_category
Very Far    34.23
Far         32.81
Near        18.64
Medium      13.83
Unknown      0.49
Name: count, dtype: float64

========== DISTANCE VS LOGISTICS ==========
distance_category  order_items  average_distance_km  average_delivery_days  delayed_items  average_freight  average_review_score  delayed_percentage
              Far        36958               430.09                  12.07         2347.0            19.57                  4.00                6.35
           Medium        15581               2

## 13. EDA Analysis

In [14]:
# ============================================
# STEP 32.13 — TRANSIT SPEED ANALYSIS
# ============================================

transit_analysis = (
    order_items_complete
    .groupby("transit_speed", observed=True)
    .agg(
        order_items=("order_id", "count"),
        average_transit_days=("transit_days", "mean"),
        average_review_score=("review_score_mean", "mean"),
        delayed_items=("is_delayed", "sum"),
        average_delivery_days=("delivery_days", "mean")
    )
    .reset_index()
)

transit_analysis["delayed_percentage"] = (
    transit_analysis["delayed_items"]
    / transit_analysis["order_items"]
    * 100
)

print("========== TRANSIT SPEED DISTRIBUTION ==========")

print(
    order_items_complete["transit_speed"]
    .value_counts()
)

print("\n========== TRANSIT SPEED VS CUSTOMER EXPERIENCE ==========")

print(
    transit_analysis
    .round(2)
    .to_string(index=False)
)

========== TRANSIT SPEED DISTRIBUTION ==========
transit_speed
Slow       56235
Normal     34975
Fast       18749
Unknown     2691
Name: count, dtype: int64

========== TRANSIT SPEED VS CUSTOMER EXPERIENCE ==========
transit_speed  order_items  average_transit_days  average_review_score  delayed_items  average_delivery_days  delayed_percentage
         Fast        18749                  1.66                  4.30          253.0                   4.45                1.35
       Normal        34975                  5.08                  4.23          408.0                   7.76                1.17
         Slow        56235                 14.27                  3.92         6601.0                  17.19               11.74
      Unknown         2691                   NaN                  1.95            3.0                   6.80                0.11


## 14. EDA Analysis

In [15]:
# ============================================
# STEP 32.14 — MULTI-ITEM & ORDER VALUE ANALYSIS
# ============================================

order_type_analysis = (
    order_items_complete
    .groupby("is_multi_item_order", observed=True)
    .agg(
        order_items=("order_id", "count"),
        average_order_item_count=("order_item_count", "mean"),
        average_order_total_value=("order_total_value", "mean"),
        average_delivery_days=("delivery_days", "mean"),
        average_review_score=("review_score_mean", "mean"),
        delayed_items=("is_delayed", "sum")
    )
    .reset_index()
)

order_type_analysis["delayed_percentage"] = (
    order_type_analysis["delayed_items"]
    / order_type_analysis["order_items"]
    * 100
)

print("========== SINGLE VS MULTI-ITEM ORDERS ==========")

print(
    order_type_analysis
    .round(2)
    .to_string(index=False)
)

print("\n========== ORDER ITEM COUNT SUMMARY ==========")

print(
    order_items_complete["order_item_count"]
    .describe()
    .round(2)
)

print("\n========== ORDER TOTAL VALUE SUMMARY ==========")

print(
    order_items_complete["order_total_value"]
    .describe()
    .round(2)
)

========== SINGLE VS MULTI-ITEM ORDERS ==========
 is_multi_item_order  order_items  average_order_item_count  average_order_total_value  average_delivery_days  average_review_score  delayed_items  delayed_percentage
                   0        88863                      1.00                     150.75                  12.18                  4.16         6025.0                6.78
                   1        23787                      2.87                     290.44                  11.39                  3.55         1240.0                5.21

========== ORDER ITEM COUNT SUMMARY ==========
count    112650.00
mean          1.40
std           1.12
min           1.00
25%           1.00
50%           1.00
75%           1.00
max          21.00
Name: order_item_count, dtype: float64

========== ORDER TOTAL VALUE SUMMARY ==========
count    112650.00
mean        180.24
std         272.83
min           9.59
25%          65.62
50%         114.44
75%         195.34
max       13664.08
Name: ord

## 15. EDA Analysis

In [16]:
# ============================================
# STEP 32.15 — CORRELATION & FEATURE RELATIONSHIPS
# ============================================

numeric_features = [
    "price",
    "freight_value",
    "delivery_days",
    "estimated_delivery_days",
    "delivery_delay_days",
    "payment_total_value",
    "order_item_count",
    "order_total_value",
    "approval_days",
    "shipping_preparation_days",
    "transit_days",
    "seller_customer_distance_km",
    "freight_to_price_ratio",
    "product_volume_cm3",
    "review_score_mean"
]

correlation_matrix = (
    order_items_complete[numeric_features]
    .corr()
)

print("========== CORRELATION WITH DELIVERY DELAY ==========")

print(
    correlation_matrix["delivery_delay_days"]
    .sort_values(ascending=False)
    .round(3)
)

print("\n========== CORRELATION WITH REVIEW SCORE ==========")

print(
    correlation_matrix["review_score_mean"]
    .sort_values(ascending=False)
    .round(3)
)

========== CORRELATION WITH DELIVERY DELAY ==========
delivery_delay_days            1.000
delivery_days                  0.597
transit_days                   0.571
shipping_preparation_days      0.193
approval_days                  0.042
product_volume_cm3             0.012
price                         -0.004
order_total_value             -0.021
payment_total_value           -0.021
freight_to_price_ratio        -0.029
order_item_count              -0.035
freight_value                 -0.040
seller_customer_distance_km   -0.077
review_score_mean             -0.229
estimated_delivery_days       -0.511
Name: delivery_delay_days, dtype: float64

========== CORRELATION WITH REVIEW SCORE ==========
review_score_mean              1.000
price                         -0.004
approval_days                 -0.026
product_volume_cm3            -0.031
freight_value                 -0.036
freight_to_price_ratio        -0.037
seller_customer_distance_km   -0.053
estimated_delivery_days       -0.064


## 3.16 Return and Complaint Analysis

### Part 1

In [17]:
# ============================================
# STEP 32.16 — RETURN & COMPLAINT ANALYSIS
# PART 1: NCH COMPLAINT DATA INSPECTION
# ============================================

print("========== NCH COMPLAINT DATA ==========")

print("\nShape:")
print(df_complaints_clean.shape)

print("\nColumns:")
print(df_complaints_clean.columns.tolist())

print("\nData:")
print(
    df_complaints_clean.to_string(index=False)
)

print("\nData Types:")
print(
    df_complaints_clean.dtypes
)

========== NCH COMPLAINT DATA ==========

Shape:
(11, 3)

Columns:
['Sl. No.', 'Nature of Grievances', 'Number of Grievances Received on NCH']

Data:
 Sl. No.                              Nature of Grievances  Number of Grievances Received on NCH
       1                            Deficiency in Services                                 59723
       2                         Delivery of Wrong Product                                 54563
       3            Delivery of Defective / Damage Product                                 53285
       4                          Paid amount not refunded                                 50997
       5                           Non-Delivery of Product                                 42781
       6                      Delay in Delivery of Product                                 27394
       7 Replacement / Refund not provided - As per policy                                 19020
       8    Amount Debited but Not Credited to Beneficiary                

### Part 2

In [18]:
# ============================================
# STEP 32.16 — RETURN & COMPLAINT ANALYSIS
# PART 2: COMPLAINT DISTRIBUTION
# ============================================

complaint_analysis = df_complaints_clean.copy()

# Calculate total grievances
total_grievances = complaint_analysis[
    "Number of Grievances Received on NCH"
].sum()

# Calculate percentage
complaint_analysis["grievance_percentage"] = (
    complaint_analysis["Number of Grievances Received on NCH"]
    / total_grievances
    * 100
)

# Sort by number of grievances
complaint_analysis = complaint_analysis.sort_values(
    by="Number of Grievances Received on NCH",
    ascending=False
).reset_index(drop=True)

print("========== TOTAL GRIEVANCES ==========")
print(f"Total: {total_grievances:,}")

print("\n========== COMPLAINT DISTRIBUTION ==========")

print(
    complaint_analysis[
        [
            "Nature of Grievances",
            "Number of Grievances Received on NCH",
            "grievance_percentage"
        ]
    ]
    .round(2)
    .to_string(index=False)
)

========== TOTAL GRIEVANCES ==========
Total: 397,333

========== COMPLAINT DISTRIBUTION ==========
                             Nature of Grievances  Number of Grievances Received on NCH  grievance_percentage
                           Deficiency in Services                                 59723                 15.03
                        Delivery of Wrong Product                                 54563                 13.73
           Delivery of Defective / Damage Product                                 53285                 13.41
                                           Others                                 53265                 13.41
                         Paid amount not refunded                                 50997                 12.83
                          Non-Delivery of Product                                 42781                 10.77
                     Delay in Delivery of Product                                 27394                  6.89
Replacement / Refund

### Part 3

In [19]:
# ============================================
# STEP 32.16 — RETURN & COMPLAINT ANALYSIS
# PART 3: RETURNIQ-RELEVANT CATEGORIES
# ============================================

returniq_categories = [
    "Delivery of Wrong Product",
    "Delivery of Defective / Damage Product",
    "Non-Delivery of Product",
    "Delay in Delivery of Product",
    "Replacement / Refund not provided - As per policy",
    "Product / Product Accessories Missing"
]

refund_categories = [
    "Paid amount not refunded",
    "Replacement / Refund not provided - As per policy"
]

delivery_categories = [
    "Delivery of Wrong Product",
    "Delivery of Defective / Damage Product",
    "Non-Delivery of Product",
    "Delay in Delivery of Product",
    "Product / Product Accessories Missing"
]

# Total grievances
total_grievances = (
    df_complaints_clean[
        "Number of Grievances Received on NCH"
    ].sum()
)

# ReturnIQ-related grievances
returniq_total = (
    df_complaints_clean[
        df_complaints_clean["Nature of Grievances"]
        .isin(returniq_categories)
    ]["Number of Grievances Received on NCH"]
    .sum()
)

# Refund/replacement grievances
refund_total = (
    df_complaints_clean[
        df_complaints_clean["Nature of Grievances"]
        .isin(refund_categories)
    ]["Number of Grievances Received on NCH"]
    .sum()
)

# Delivery/product grievances
delivery_total = (
    df_complaints_clean[
        df_complaints_clean["Nature of Grievances"]
        .isin(delivery_categories)
    ]["Number of Grievances Received on NCH"]
    .sum()
)

print("========== RETURNIQ-RELEVANT COMPLAINTS ==========")

print(f"Total NCH grievances: {total_grievances:,}")

print(
    f"ReturnIQ-relevant categories: "
    f"{returniq_total:,} "
    f"({returniq_total / total_grievances * 100:.2f}%)"
)

print(
    f"Delivery/Product issue categories: "
    f"{delivery_total:,} "
    f"({delivery_total / total_grievances * 100:.2f}%)"
)

print(
    f"Refund/Replacement categories: "
    f"{refund_total:,} "
    f"({refund_total / total_grievances * 100:.2f}%)"
)

print("\n========== INCLUDED CATEGORIES ==========")

for category in returniq_categories:
    count = df_complaints_clean.loc[
        df_complaints_clean["Nature of Grievances"] == category,
        "Number of Grievances Received on NCH"
    ].sum()

    percentage = count / total_grievances * 100

    print(
        f"{category}: "
        f"{count:,} ({percentage:.2f}%)"
    )

========== RETURNIQ-RELEVANT COMPLAINTS ==========
Total NCH grievances: 397,333
ReturnIQ-relevant categories: 212,120 (53.39%)
Delivery/Product issue categories: 193,100 (48.60%)
Refund/Replacement categories: 70,017 (17.62%)

========== INCLUDED CATEGORIES ==========
Delivery of Wrong Product: 54,563 (13.73%)
Delivery of Defective / Damage Product: 53,285 (13.41%)
Non-Delivery of Product: 42,781 (10.77%)
Delay in Delivery of Product: 27,394 (6.89%)
Replacement / Refund not provided - As per policy: 19,020 (4.79%)
Product / Product Accessories Missing: 15,077 (3.79%)


### Part 4

In [20]:
# ============================================
# STEP 32.16 — RETURN & COMPLAINT ANALYSIS
# PART 4: ONLINE RETAIL II RETURN-SIGNAL ANALYSIS
# ============================================

print("========== ONLINE RETAIL II RETURN-SIGNAL ANALYSIS ==========")

# Check required columns
required_columns = [
    "Quantity",
    "Price",
    "Customer ID",
    "Country"
]

print("\nAvailable required columns:")

for col in required_columns:
    print(f"{col}: {col in df_online_retail_clean.columns}")


# ============================================
# BASIC QUANTITY ANALYSIS
# ============================================

print("\n========== QUANTITY ANALYSIS ==========")

total_records = len(df_online_retail_clean)

negative_quantity_records = (
    df_online_retail_clean["Quantity"] < 0
).sum()

positive_quantity_records = (
    df_online_retail_clean["Quantity"] > 0
).sum()

zero_quantity_records = (
    df_online_retail_clean["Quantity"] == 0
).sum()

print(f"Total transaction records: {total_records:,}")

print(
    f"Negative-quantity records: "
    f"{negative_quantity_records:,} "
    f"({negative_quantity_records / total_records * 100:.2f}%)"
)

print(
    f"Positive-quantity records: "
    f"{positive_quantity_records:,} "
    f"({positive_quantity_records / total_records * 100:.2f}%)"
)

print(
    f"Zero-quantity records: "
    f"{zero_quantity_records:,} "
    f"({zero_quantity_records / total_records * 100:.2f}%)"
)


# ============================================
# RETURN-SIGNAL QUANTITY
# ============================================

negative_quantity_total = (
    df_online_retail_clean.loc[
        df_online_retail_clean["Quantity"] < 0,
        "Quantity"
    ].sum()
)

print("\n========== NEGATIVE QUANTITY ==========")

print(
    f"Total negative quantity: "
    f"{negative_quantity_total:,}"
)


# ============================================
# RETURN-SIGNAL TRANSACTION VALUE
# ============================================

if "Price" in df_online_retail_clean.columns:

    negative_records = df_online_retail_clean[
        df_online_retail_clean["Quantity"] < 0
    ].copy()

    negative_records["Transaction_Value"] = (
        negative_records["Quantity"].abs()
        * negative_records["Price"]
    )

    print(
        f"Estimated value represented by "
        f"negative-quantity records: "
        f"{negative_records['Transaction_Value'].sum():,.2f}"
    )


# ============================================
# CUSTOMER-LEVEL RETURN SIGNAL
# ============================================

if "Customer ID" in df_online_retail_clean.columns:

    customer_return_signals = (
        df_online_retail_clean[
            df_online_retail_clean["Quantity"] < 0
        ]
        .groupby("Customer ID")
        .size()
        .sort_values(ascending=False)
    )

    print("\n========== TOP CUSTOMERS BY RETURN SIGNAL ==========")

    print(
        customer_return_signals.head(10)
    )


# ============================================
# COUNTRY-LEVEL RETURN SIGNAL
# ============================================

if "Country" in df_online_retail_clean.columns:

    country_return_signals = (
        df_online_retail_clean[
            df_online_retail_clean["Quantity"] < 0
        ]
        .groupby("Country")
        .size()
        .sort_values(ascending=False)
    )

    print("\n========== TOP COUNTRIES BY RETURN SIGNAL ==========")

    print(
        country_return_signals.head(10)
    )


print("\n========== INTERPRETATION ==========")

print(
    "Negative-quantity transactions are treated as "
    "return/cancellation signals rather than confirmed "
    "product returns."
)

========== ONLINE RETAIL II RETURN-SIGNAL ANALYSIS ==========

Available required columns:
Quantity: True
Price: True
Customer ID: True
Country: True

========== QUANTITY ANALYSIS ==========
Total transaction records: 514,912
Negative-quantity records: 10,181 (1.98%)
Positive-quantity records: 504,731 (98.02%)
Zero-quantity records: 0 (0.00%)

========== NEGATIVE QUANTITY ==========
Total negative quantity: -213,289
Estimated value represented by negative-quantity records: 629,808.65

========== TOP CUSTOMERS BY RETURN SIGNAL ==========
Customer ID
15311.0    171
14911.0    139
17511.0    110
14680.0    103
13798.0    102
14527.0     97
12471.0     89
12472.0     86
14277.0     83
13091.0     74
dtype: int64

========== TOP COUNTRIES BY RETURN SIGNAL ==========
Country
United Kingdom          8770
Germany                  468
France                   240
EIRE                     209
Channel Islands           85
Japan                     60
Spain                     43
Portugal         

### Part 5

In [21]:
# ============================================
# STEP 32.16 — RETURN & COMPLAINT ANALYSIS
# PART 5: COMBINE RETURN & COMPLAINT FINDINGS
# ============================================

print("========== RETURNIQ RETURN & COMPLAINT SUMMARY ==========")


# ============================================
# NCH COMPLAINT FINDINGS
# ============================================

print("\n========== NCH COMPLAINT FINDINGS ==========")

print(
    "The NCH dataset provides external evidence "
    "of common consumer complaint categories."
)

print(
    "These complaints are used as supporting evidence "
    "for ReturnIQ-related return and customer problems."
)


# ============================================
# ONLINE RETAIL II FINDINGS
# ============================================

print("\n========== ONLINE RETAIL II FINDINGS ==========")

total_transactions = len(df_online_retail_clean)

negative_transactions = (
    df_online_retail_clean["Quantity"] < 0
).sum()

negative_percentage = (
    negative_transactions / total_transactions * 100
)

negative_quantity = (
    df_online_retail_clean.loc[
        df_online_retail_clean["Quantity"] < 0,
        "Quantity"
    ].sum()
)

print(
    f"Total transaction records: "
    f"{total_transactions:,}"
)

print(
    f"Return/cancellation signal records: "
    f"{negative_transactions:,} "
    f"({negative_percentage:.2f}%)"
)

print(
    f"Total negative quantity: "
    f"{negative_quantity:,}"
)


# ============================================
# RETURN SIGNAL BY COUNTRY
# ============================================

print("\n========== RETURN SIGNAL BY COUNTRY ==========")

country_return_signals = (
    df_online_retail_clean[
        df_online_retail_clean["Quantity"] < 0
    ]
    .groupby("Country")
    .size()
    .sort_values(ascending=False)
)

print(
    country_return_signals.head(10)
)


# ============================================
# RETURN SIGNAL BY CUSTOMER
# ============================================

print("\n========== RETURN SIGNAL BY CUSTOMER ==========")

customer_return_signals = (
    df_online_retail_clean[
        df_online_retail_clean["Quantity"] < 0
    ]
    .groupby("Customer ID")
    .size()
    .sort_values(ascending=False)
)

print(
    customer_return_signals.head(10)
)


# ============================================
# FINAL BUSINESS FINDINGS
# ============================================

print("\n========== FINAL RETURNIQ FINDINGS ==========")

print(
    "1. Negative-quantity transactions provide "
    "a measurable return/cancellation signal."
)

print(
    "2. Return signals are concentrated among "
    "specific customers and countries."
)

print(
    "3. NCH complaint categories provide external "
    "evidence of delivery, product, refund, and "
    "replacement-related customer problems."
)

print(
    "4. These findings support the need for an "
    "AI-based return intelligence system."
)

print(
    "5. ReturnIQ can use historical transaction and "
    "customer-related features to identify patterns "
    "associated with return behavior."
)

print(
    "\nNote: Negative quantity is treated as a "
    "return/cancellation signal, not a confirmed "
    "product return."
)

========== RETURNIQ RETURN & COMPLAINT SUMMARY ==========

========== NCH COMPLAINT FINDINGS ==========
The NCH dataset provides external evidence of common consumer complaint categories.
These complaints are used as supporting evidence for ReturnIQ-related return and customer problems.

========== ONLINE RETAIL II FINDINGS ==========
Total transaction records: 514,912
Return/cancellation signal records: 10,181 (1.98%)
Total negative quantity: -213,289

========== RETURN SIGNAL BY COUNTRY ==========
Country
United Kingdom          8770
Germany                  468
France                   240
EIRE                     209
Channel Islands           85
Japan                     60
Spain                     43
Portugal                  40
Netherlands               39
United Arab Emirates      33
dtype: int64

========== RETURN SIGNAL BY CUSTOMER ==========
Customer ID
15311.0    171
14911.0    139
17511.0    110
14680.0    103
13798.0    102
14527.0     97
12471.0     89
12472.0     86
142

## 3.17 Final EDA Feature Review

In [22]:
# ============================================
# STEP 32.17 — FINAL EDA
# PART 1: CHECK AVAILABLE FEATURES
# ============================================

print("========== AVAILABLE FEATURES ==========")

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

========== AVAILABLE FEATURES ==========
1. order_id
2. order_item_id
3. product_id
4. seller_id
5. shipping_limit_date
6. price
7. freight_value
8. product_category_name
9. product_weight_g
10. product_length_cm
11. product_height_cm
12. product_width_cm
13. seller_zip_code_prefix
14. seller_city
15. seller_state
16. customer_id
17. order_status
18. order_purchase_timestamp
19. order_approved_at
20. order_delivered_carrier_date
21. order_delivered_customer_date
22. order_estimated_delivery_date
23. customer_unique_id
24. customer_zip_code_prefix
25. customer_city
26. customer_state
27. payment_total_value
28. payment_count
29. payment_max_installments
30. payment_types
31. review_score_mean
32. review_score_min
33. review_score_count
34. review_comments
35. product_category_name_english
36. customer_latitude
37. customer_longitude
38. seller_latitude
39. seller_longitude
40. review_available
41. payment_available
42. carrier_delivery_available
43. customer_delivery_available
44. appro

## EDA Complete

The observed patterns from this notebook will be used to select appropriate features and define
the ML problem. Target selection must be based on the available labels and should not assume that
historical complaint categories are individual order-level labels.